- ### Running cpp files

1. [In Jupyter via custom magic %%cpp](#in-jupyter-via-custom-magic-cpp)

2. [In Jupyter via magic %%writefile](#in-jupyter-via-magic-writefile)

3. [Cpp file via terminal](#cpp-file-via-terminal)

4. [Cpp file via shortcut](#cpp-file-via-shortcut)
---

- ### In Jupyter via custom magic %%cpp

**setup for oneline command %%cpp**

In [1]:
# Setup for oneline command %%cpp
import os, tempfile, subprocess
from IPython.core.magic import register_cell_magic
import shlex

@register_cell_magic
def cpp(line, cell):
    """
    Usage:
    %%cpp -i "input for cin" -- arg1 arg2 ...
    """
    tokens = shlex.split(line)
    input_data = None
    run_args = []

    # Parse stdin input
    if "-i" in tokens:
        idx = tokens.index("-i")
        if idx + 1 < len(tokens):
            input_data = tokens[idx + 1]

    # Parse program arguments after --
    if "--" in tokens:
        idx = tokens.index("--")
        run_args = tokens[idx + 1:]

    # Write temp C++ file
    with tempfile.NamedTemporaryFile(suffix=".cpp", delete=False, mode="w") as tmp_cpp:
        tmp_cpp.write(cell)
        cpp_path = tmp_cpp.name
    exe_path = cpp_path[:-4] + ".exe"

    try:
        # Compile
        compile_proc = subprocess.run(
            ["g++", "-std=c++17", "-O2", "-Wall", cpp_path, "-o", exe_path],
            capture_output=True,
            text=True
        )
        if compile_proc.returncode != 0:
            print("❌ Compilation failed:\n", compile_proc.stderr)
            return

        # Run program
        run_proc = subprocess.run(
            [exe_path] + run_args,
            input=input_data,      # feed stdin here
            capture_output=True,
            text=True
        )
        if run_proc.stdout:
            print(run_proc.stdout, end="")
        if run_proc.stderr:
            print("⚠️ Runtime error:\n", run_proc.stderr)

    finally:
        for f in (cpp_path, exe_path):
            try: os.remove(f)
            except: pass

**using magic %%cpp**

In [ ]:
%%cpp
#include <iostream> //head file for input/output operations
using namespace std; //std namespace to avoid prefixing std:: to standard functions

int main() { //main function, entry point of the program
    cout << "Hello from custom %%cpp magic!" << endl;
}

Hello from custom %%cpp magic!


**Select Cell Language Mode:**

1. Choose Python when running

2. Choose Auto detect when displaying

**with stdin**

In [ ]:
%%cpp -i "8 6"
#include <iostream>
using namespace std;
int main() {
    int a, b;
    cin >> a >> b;
    cout << a + b << '\n';
    return 0;
}

14


**with command-line arguments**

In [ ]:
%%cpp -- 7 8
#include <iostream>

int main(int argc, char* argv[]) { //receive command line arguments argument count and array of arguments
    int a = std::stoi(argv[1]);
    int b = std::stoi(argv[2]);
    std::cout << a * b << std::endl; //declare std:: if not using namespace std;
}

56


---

- ### In Jupyter via magic %%writefile

**create a subfolder**

In [93]:
!mkdir compiled

**write cpp file**

In [103]:
%%writefile compiled/test.cpp
#include <iostream>
using namespace std; 
int main() { //main function, entry point of the program
    cout << "Hello world!" << endl;
}

Overwriting compiled/test.cpp


**compile cpp file and generate exe file then run it**

In [104]:
!g++ compiled/test.cpp -o compiled/test.exe
!compiled\test.exe

Hello world!


**with stdin**

In [126]:
%%writefile compiled/test2.cpp
#include <iostream>
using namespace std;
int main() {
    int a, b;
    cin >> a >> b;
    cout << a + b << '\n';
    return 0;
}

Overwriting compiled/test2.cpp


In [127]:
!g++ compiled/test2.cpp -o compiled/test2.exe
!echo 5 7 | compiled\test2.exe

12


**with command-line arguments**

In [124]:
%%writefile compiled/test3.cpp
#include <iostream>

int main(int argc, char* argv[]) { //receive command line arguments argument count and array of arguments
    int a = std::stoi(argv[1]);
    int b = std::stoi(argv[2]);
    std::cout << a * b << std::endl; //declare std:: if not using namespace std;
}

Overwriting compiled/test3.cpp


In [125]:
!g++ compiled/test3.cpp -o compiled/test3.exe
!compiled\test3.exe 3 5

15


---

- ### Cpp file via terminal

**using g++ command with parameters**

```bash
g++ filename.cpp -o outputname
./outputname.exe parameter1 parameter2
```

---

- ### Cpp file via shortcut

**compile & execute: press ```ctrl+shift+b```**